# qrace report mode — comparative grid, Shapley budgets, and the crossover curve

This notebook runs a grid of (option, target) cases across noise levels, then digs into
the two analytical views that make the verdict interpretable:

1. the **report table + figure** (noise-aware estimates vs the classical reference),
2. the **Shapley error budget** per noise level,
3. the **quantum-vs-classical crossover** as a function of target accuracy.

Note: each grid cell runs real (noisy) Aer simulations — the grid below is kept small
and coarse so the whole notebook runs in a couple of minutes.

In [ ]:
from qrace import AnalysisTarget, NoiseProfile, OptionSpec, QiskitBackend, run_report

call = OptionSpec(
    kind="european_call", spot=100.0, strike=105.0, maturity=1.0, volatility=0.2, rate=0.05
)
put = OptionSpec(
    kind="european_put", spot=100.0, strike=95.0, maturity=1.0, volatility=0.2, rate=0.05
)
cases = [(call, AnalysisTarget(target_abs_error=2.0)), (put, AnalysisTarget(target_abs_error=2.0))]
noise_levels = [
    NoiseProfile(kind="ideal"),
    NoiseProfile(kind="depolarizing", two_qubit_error=0.001),
    NoiseProfile(kind="depolarizing", two_qubit_error=0.01),
]
backend = QiskitBackend(seed=7, shots=512)

result = run_report(cases, noise_levels, backend)
result.table

In [ ]:
result.figure()

The pattern to look for: as the two-qubit error rate grows, the estimate drifts away from
the classical reference and `quantum_advantageous` flips to `False` everywhere. The table
also shows *why* — required fidelity vs what the noise level implies, and oracle calls vs
Monte Carlo samples.

## Shapley budgets across noise levels

Same option, increasing noise: watch the `decoherence` share of the error budget grow
until it dwarfs the (constant) loading and payoff-encoding biases.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qrace import analyze

target = AnalysisTarget(target_abs_error=2.0)
budgets = {}
for noise in noise_levels:
    label = noise.kind if noise.kind == "ideal" else f"depol {noise.two_qubit_error}"
    budgets[label] = analyze(call, target, backend, noise).shapley.contributions

stages = ["loading", "payoff", "qae_iters", "decoherence"]
x = np.arange(len(stages))
width = 0.8 / len(budgets)
fig, ax = plt.subplots(figsize=(8, 4))
for i, (label, contribs) in enumerate(budgets.items()):
    ax.bar(x + i * width, [contribs[s] for s in stages], width, label=label)
ax.set_xticks(x + width)
ax.set_xticklabels(stages)
ax.set_ylabel("Shapley contribution to |price error|")
ax.set_title("Error budget by stage, per noise level")
ax.legend()
plt.tight_layout()
plt.show()

## The crossover curve

Pure query-complexity comparison at equal target accuracy ε: classical Monte Carlo needs
N(ε) = ⌈(zσ/ε)²⌉ samples, amplitude estimation needs M(ε) ∝ 1/ε oracle calls. Quantum
wins in queries only left of the intersection — and remember this ignores the fidelity
gate entirely (a coherent oracle call also costs far more wall-clock than an MC sample).

In [ ]:
from qrace.classical import discounted_payoff_std
from qrace.crossover import compute

sigma = discounted_payoff_std(call)
epsilons = np.logspace(-4, 0, 60)
classical = []
quantum = []
for eps in epsilons:
    c = compute(sigma, AnalysisTarget(target_abs_error=float(eps)))
    classical.append(c.classical_mc_samples)
    quantum.append(c.quantum_oracle_calls)
breakeven = compute(sigma, AnalysisTarget(target_abs_error=0.01)).breakeven_error

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.loglog(epsilons, classical, label="classical MC samples  N(ε) ~ 1/ε²")
ax.loglog(epsilons, quantum, label="quantum oracle calls  M(ε) ~ 1/ε")
ax.axvline(breakeven, color="gray", linestyle="--", label=f"break-even ε ≈ {breakeven:.2e}")
ax.set_xlabel("target absolute error ε")
ax.set_ylabel("queries")
ax.set_title(f"Crossover for the demo call (payoff σ ≈ {sigma:.2f})")
ax.legend()
plt.tight_layout()
plt.show()

**Takeaway.** The query crossover sits at tight ε, and every step toward tighter ε deepens
the circuit and raises the required fidelity. That squeeze — advantage only where noise
kills you — is the honest core finding, and it is exactly what the `Verdict` quantifies
per case instead of asserting in general.